<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 23**
---

# **Un caso práctico: enseñar una agenda de teléfonos a un modelo de lenguaje**

En los dos notebooks anteriores vimos la teoría: el entrenamiento autorregresivo, el SFT y las técnicas PEFT. En este vamos a juntarlo todo en un experimento pequeño, pero muy revelador, sobre cómo se entrena un modelo para un **dominio específico**.

El dominio será el más sencillo que se nos ha ocurrido: una **agenda de teléfonos inventada**. Queremos un modelo al que se le pueda preguntar «¿Cuál es el teléfono de Federico?» y responda con el número correcto.

Parece trivial, pero esconde la pregunta central de la adaptación a un dominio:

> **¿Puede un modelo responder preguntas sobre algo que solo ha leído en texto plano, y por lo que nadie le ha preguntado nunca durante el entrenamiento?**

Es exactamente la situación real. Una empresa tiene miles de documentos (texto plano), pero solo puede permitirse escribir a mano unos cuantos ejemplos de pregunta y respuesta. Lo que necesita es que el modelo aprenda **los hechos** de los documentos y **la forma de responder** de los ejemplos, y que sepa combinar ambas cosas.

Veremos tres formas de entrenar con los mismos datos, y que solo una de ellas lo consigue.

**Qué vas a aprender:**

1. Cómo se diseña un experimento para saber si el modelo ha *aprendido* o solo ha *memorizado respuestas*.
2. Qué aporta el entrenamiento autorregresivo y qué aporta el SFT, vistos por separado.
3. Por qué encadenarlos (primero uno y después el otro) puede fallar.
4. Por qué **alternarlos** funciona mejor.

> **Nota.** El entrenamiento utiliza un modelo de 8.000 millones de parámetros cuantizado a 4 bits y necesita una GPU con CUDA. Las celdas de exploración de datos se pueden ejecutar en cualquier ordenador. Las salidas de los modelos que se muestran en el texto son las que se obtuvieron al ejecutar los notebooks originales de la carpeta `code/auto_sft`.

## **1. El dominio: una agenda inventada**

¿Por qué teléfonos inventados? Porque así tenemos una garantía: **el modelo no puede conocerlos de antemano**. Si al final responde con el número correcto, solo puede haberlo aprendido en nuestro entrenamiento. Con un dominio real (medicina, derecho) nunca sabríamos cuánto sabía ya el modelo.

Tenemos dos ficheros, uno para cada tipo de entrenamiento:

- `sentences_train.txt`: **texto plano**. Frases que enuncian un hecho. Se utilizará en el entrenamiento autorregresivo.
- `questions_train.jsonl`: **preguntas y respuestas** con la plantilla `[INST] ... [/INST]`. Se utilizará en el SFT.

In [1]:
import json

DATA_DIR = "../code/auto_sft/dataset"

with open(f"{DATA_DIR}/sentences_train.txt", encoding="utf-8") as f:
    frases = [linea.strip() for linea in f if linea.strip()]

with open(f"{DATA_DIR}/questions_train.jsonl", encoding="utf-8") as f:
    preguntas = [json.loads(linea)["text"] for linea in f if linea.strip()]

print(f"TEXTO PLANO: {len(frases)} frases")
for frase in frases[:6]:
    print("   ", frase)

print(f"\nPREGUNTAS Y RESPUESTAS: {len(preguntas)} ejemplos")
for pregunta in preguntas[:6]:
    print("   ", pregunta)

TEXTO PLANO: 150 frases
    El número de Jose Francisco es 606919756.
    El número de Ariadna es 619609877.
    Jose Enrique tiene el teléfono 616171519.
    El número de Iñaki es 619305624.
    El teléfono de Luis Alberto es 928871713.
    El número de Jesus Manuel es 606523164.

PREGUNTAS Y RESPUESTAS: 50 ejemplos
    <s>[INST] ¿Cuál es el número de Jose Francisco? [/INST] El número de Jose Francisco es el 606919756. </s>
    <s>[INST] Dame el teléfono de Ariadna [/INST] El número de teléfono de Ariadna es el 619609877. </s>
    <s>[INST] ¿Qué teléfono tiene Jose Enrique? [/INST] El teléfono de Jose Enrique es el 616171519. </s>
    <s>[INST] ¿Qué teléfono tiene Iñaki? [/INST] El teléfono de Iñaki es el 619305624. </s>
    <s>[INST] Dame el teléfono de Luis Alberto [/INST] El número de teléfono de Luis Alberto es el 928871713. </s>
    <s>[INST] ¿Cuál es el teléfono de Jesus Manuel? [/INST] El teléfono de Jesus Manuel es el 606523164. </s>


Fíjate en dos detalles del diseño de los datos:

- **Las frases no son todas iguales.** El mismo tipo de hecho se enuncia de varias maneras («El número de X es...», «X tiene el teléfono...», «X tiene el tlf...»). Las preguntas también varían («¿Cuál es el teléfono de X?», «Dame el teléfono de X»...). Así el modelo no puede apoyarse en una única plantilla fija.
- **La forma de la respuesta no aparece en el texto plano.** En las frases nunca hay una pregunta, ni los marcadores `[INST]`, ni una respuesta redactada como «El teléfono de X es el...». Eso solo está en el segundo fichero.

Ahora viene lo importante. Vamos a reconstruir la agenda a partir de las frases y a comprobar **por qué personas se pregunta** en el segundo fichero.

In [2]:
import re

PLANTILLAS = [
    r"El número de (.+) es (\d{9})\.",
    r"El teléfono de (.+) es (\d{9})\.",
    r"(.+) tiene el teléfono (\d{9})\.",
    r"(.+) tiene el número (\d{9})\.",
    r"(.+) tiene el tlf (\d{9})\.",
]

# La agenda completa: nombre -> teléfono, extraída del texto plano
agenda = {}
for frase in frases:
    for plantilla in PLANTILLAS:
        m = re.fullmatch(plantilla, frase)
        if m:
            agenda[m.group(1)] = m.group(2)
            break

# Los teléfonos que aparecen en alguna pregunta del fichero de SFT
telefonos_sft = {re.search(r"\d{9}", p).group() for p in preguntas}

vistas_en_sft = [nombre for nombre, tlf in agenda.items() if tlf in telefonos_sft]
solo_en_texto = [nombre for nombre, tlf in agenda.items() if tlf not in telefonos_sft]

print(f"Personas en la agenda (texto plano):     {len(agenda)}")
print(f"Personas por las que se pregunta en SFT: {len(vistas_en_sft)}")
print(f"Personas que SOLO aparecen en el texto:  {len(solo_en_texto)}")

print("\nAlgunas personas que solo aparecen en el texto:")
for nombre in ["Pilar", "Emilia", "Salvador", "Federico"]:
    print(f"    {nombre:10s} {agenda[nombre]}   ¿tiene pregunta en SFT? {'sí' if nombre in vistas_en_sft else 'no'}")

Personas en la agenda (texto plano):     150
Personas por las que se pregunta en SFT: 50
Personas que SOLO aparecen en el texto:  100

Algunas personas que solo aparecen en el texto:
    Pilar      616280808   ¿tiene pregunta en SFT? no
    Emilia     619492312   ¿tiene pregunta en SFT? no
    Salvador   606638451   ¿tiene pregunta en SFT? no
    Federico   928204525   ¿tiene pregunta en SFT? no


Este es el corazón del experimento:

<div align="center">
    <img src="./imgs/agenda_experimento.svg" width="850px">
</div>

- Las **150 personas** aparecen en el texto plano.
- Solo de **50** de ellas hay, además, un ejemplo de pregunta y respuesta.
- Las otras **100** son nuestro **examen**. El modelo ha leído su teléfono una vez, en una frase suelta, y nadie le ha enseñado a responder una pregunta sobre ellas.

Si preguntásemos por una de las 50 personas del SFT, un acierto no demostraría nada: el modelo podría haber memorizado la pareja pregunta–respuesta tal cual. En cambio, si acierta con Pilar, Salvador o Federico, es que ha hecho algo más interesante: ha **recuperado un hecho** aprendido del texto y lo ha **expresado en el formato** aprendido de los ejemplos. Eso es justo lo que se le pide a un modelo de dominio específico.

Apunta estos tres teléfonos, porque los vamos a necesitar:

| Persona | Teléfono real | Frase en el texto plano |
|---|---|---|
| Pilar | 616280808 | `Pilar tiene el número 616280808.` |
| Salvador | 606638451 | `El teléfono de Salvador es 606638451.` |
| Federico | 928204525 | `Federico tiene el número 928204525.` |

## **2. El modelo y las herramientas**

Utilizaremos **Llama 3.1 de 8.000 millones de parámetros**, cuantizado a 4 bits, y lo adaptaremos con **LoRA** (notebook 22): los pesos originales quedan congelados y solo se entrenan las matrices de bajo rango, que aquí se añaden a todas las proyecciones de la atención y a las capas *feed-forward*. Es, en esencia, la receta de QLoRA.

Observa en el código que el entrenamiento autorregresivo y el SFT comparten **todo**: el mismo modelo, el mismo `Trainer`, la misma función de pérdida (predecir el siguiente token). Lo único que cambia es **el fichero de datos**. Es lo que decíamos en el notebook 21: el SFT no es un mecanismo de aprendizaje distinto, sino el mismo mecanismo con otros datos.

Una observación sobre la plantilla: los ejemplos usan los marcadores `<s>[INST] ... [/INST] ... </s>`, que son los de Mistral, no los de Llama. Para este modelo son simple texto, y no pasa nada: precisamente el SFT le enseñará qué significan.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# Los dos conjuntos de datos, tokenizados exactamente de la misma manera
dataset_autoreg = load_dataset("text", data_files={"train": f"{DATA_DIR}/sentences_train.txt"})["train"]
dataset_sft = load_dataset("json", data_files=f"{DATA_DIR}/questions_train.jsonl", split="train")

tokenized_autoreg = dataset_autoreg.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_sft = dataset_sft.map(tokenize_function, batched=True, remove_columns=["text"])

# mlm=False -> modelado causal: la etiqueta de cada posición es el token siguiente
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
def nuevo_modelo_con_lora(origen=base_model_name, lora_alpha=32):
    # Carga un modelo y le añade un adaptador LoRA recién inicializado
    model = AutoModelForCausalLM.from_pretrained(origen)
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,                    # rango de LoRA
        lora_alpha=lora_alpha,  # factor de escala
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def crear_trainer(model, dataset, epochs):
    # El mismo Trainer sirve para el entrenamiento autorregresivo y para el SFT
    args = TrainingArguments(
        output_dir="./results",
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        learning_rate=1e-4,
        fp16=True,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
    )
    return Trainer(model=model, args=args, train_dataset=dataset, data_collator=data_collator)


def generar(model, prompt, max_length=45):
    # Continúa el texto del prompt (temperatura casi cero: prácticamente determinista)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=max_length, do_sample=True,
                             top_k=50, top_p=0.95, temperature=0.01)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def preguntar(model, pregunta):
    # Hace una pregunta con la misma plantilla que se usó en el SFT
    return generar(model, f"<s>[INST] {pregunta} [/INST]")

## **3. Experimento A: solo entrenamiento autorregresivo**

Empezamos por lo más directo: darle a leer la agenda. Entrenamos 30 épocas sobre las frases de texto plano, sin ninguna pregunta.

**¿Qué esperamos?** Que el modelo aprenda a *continuar* frases de la agenda. No esperamos que sepa responder preguntas, porque no ha visto ninguna.

In [ ]:
model_A = nuevo_modelo_con_lora()
crear_trainer(model_A, tokenized_autoreg, epochs=30).train()

In [ ]:
# 1) Le damos el comienzo de una frase que está, literalmente, en el texto de entrenamiento
print(generar(model_A, "Pilar tiene el número", max_length=25))

# 2) Le hacemos preguntas, algo que nunca ha visto
print(generar(model_A, "¿Qué número de tlf tiene Pilar?", max_length=25))
print(generar(model_A, "¿Qué número de tlf tiene Emilia?", max_length=35))

**Resultado obtenido:**

```
Pilar tiene el número 34. ¿Cuál es el número de su hermano? Pilar tiene un

¿Qué número de tlf tiene Pilar? Pilar tiene un tlf que empieza con 616 y

¿Qué número de tlf tiene Emilia? ¿Es un número de teléfono de Madrid? ¿Es un número de teléfono de Barcelona? ¿Es un número
```

**¿Qué ha pasado?**

- **El modelo se comporta como un continuador de texto**, no como un asistente. Ante una pregunta no responde: sigue escribiendo, y lo más plausible después de una pregunta resulta ser... otra pregunta. Es el comportamiento de "modelo base" que describimos en el notebook 21.
- **No sabe cuándo parar.** Nadie le ha enseñado que una respuesta termina.
- **Hay un indicio de que algo ha aprendido.** El teléfono de Pilar es 616280808, y el modelo dice que «empieza con 616». Conviene no entusiasmarse: en la agenda solo hay cuatro prefijos (606, 616, 619 y 928), así que acertar el prefijo por casualidad no es difícil. Aun así, el modelo ha dejado de hablar de cualquier cosa y habla de teléfonos con el aspecto de los nuestros.
- **El hecho no está bien fijado.** Ni siquiera completa correctamente la frase literal del entrenamiento. Ver un dato una sola vez, redactado de una sola manera, es una señal muy débil.

**Conclusión del experimento A:** el texto plano aporta *material* (el vocabulario y los hechos del dominio), pero no aporta *comportamiento*. Falta enseñarle a responder.

## **4. Experimento B: primero autorregresivo y después SFT**

La solución obvia es seguir el orden "de manual": primero que lea el texto (ya lo ha hecho en el experimento A) y después enseñarle a responder con un SFT de 10 épocas sobre las 50 preguntas.

Para ello fusionamos el adaptador LoRA del experimento A con el modelo, de modo que lo aprendido quede incorporado a los pesos, y sobre ese modelo fusionado entrenamos un adaptador nuevo con las preguntas.

**¿Qué esperamos?** Que el modelo combine las dos cosas: los 150 teléfonos que leyó en la primera fase y el formato de respuesta que aprende en la segunda.

In [ ]:
ruta_adaptador = "./modelos/agenda_autorregresivo"
ruta_fusionado = "./modelos/agenda_autorregresivo_fusionado"

# Guardamos el adaptador del experimento A y liberamos la GPU
model_A.save_pretrained(ruta_adaptador)
del model_A
torch.cuda.empty_cache()

# Fusionamos el adaptador con el modelo base: W' = W + (alpha/r)·AB
base = AutoModelForCausalLM.from_pretrained(base_model_name)
fusionado = PeftModel.from_pretrained(base, ruta_adaptador).merge_and_unload()
fusionado.save_pretrained(ruta_fusionado)
del base, fusionado
torch.cuda.empty_cache()

In [ ]:
# SFT sobre el modelo que ya ha leído la agenda
model_B = nuevo_modelo_con_lora(origen=ruta_fusionado, lora_alpha=16)
crear_trainer(model_B, tokenized_sft, epochs=10).train()

# Salvador solo aparece en el texto plano: es una de las 100 personas del examen
print(preguntar(model_B, "¿Cuál es el teléfono de Salvador?"))

**Resultado obtenido:**

```
<s>[INST] ¿Cuál es el teléfono de Salvador? [/INST] El teléfono de Salvador es el 619977905. </s>
```

A primera vista es un éxito: el modelo responde como un asistente, con una frase bien construida, un número de nueve cifras con un prefijo válido, y termina donde debe. La pérdida del SFT, además, baja sin problemas.

Pero compruébalo con la tabla del apartado 1: **el teléfono de Salvador es 606638451**. La respuesta es falsa.

Y hay algo todavía más instructivo. El número 619977905 no es inventado: **es el teléfono de Gema**, una de las 50 personas del conjunto de SFT. El modelo no ha consultado lo que leyó sobre Salvador; ha respondido con uno de los números que había visto *en posición de respuesta*.

**¿Qué ha pasado?**

- **El SFT ha enseñado la forma, pero no la conexión con los hechos.** Durante el SFT, las únicas respuestas correctas eran 50 números concretos, y lo más fácil para reducir la pérdida era memorizar esas 50 parejas. Nada obligaba al modelo a aprender la regla general «para responder, busca lo que leíste sobre esa persona».
- **Lo aprendido en la primera fase era frágil y se ha degradado.** En el experimento A vimos que los hechos estaban poco fijados. Diez épocas dedicadas en exclusiva a otras 50 personas y a otro formato los debilitan todavía más: es el **olvido catastrófico**. (La fusión del adaptador con un modelo cuantizado a 4 bits añade, además, errores de redondeo.)
- **Es una alucinación de manual.** La respuesta es fluida, tiene el formato exacto y es completamente plausible. Sin la agenda delante, sería imposible saber que es incorrecta. Un buen formato no es evidencia de una respuesta correcta.

**Conclusión del experimento B:** el modelo ha aprendido *a responder*, pero no *a responder con lo que sabe*. Las dos fases han quedado como compartimentos estancos.

## **5. Experimento C: alternando autorregresivo y SFT**

Probemos otra cosa: en lugar de hacer primero una fase y luego la otra, las **intercalamos**. Una época sobre el texto plano, una época sobre las preguntas, y se repite el ciclo. Partimos de cero (del modelo base con un adaptador nuevo) y hay un único adaptador LoRA, que comparten los dos entrenamientos.

El código es sorprendentemente simple: dos `Trainer` sobre el mismo modelo, cada uno con su conjunto de datos, dentro de un bucle.

In [ ]:
del model_B
torch.cuda.empty_cache()

model_C = nuevo_modelo_con_lora()

# Dos Trainers sobre el MISMO modelo: cada llamada a train() es una época
trainer_autoreg = crear_trainer(model_C, tokenized_autoreg, epochs=1)
trainer_sft = crear_trainer(model_C, tokenized_sft, epochs=1)

N_CICLOS = 7

for ciclo in range(N_CICLOS):
    print(f"Ciclo {ciclo + 1}: época autorregresiva (150 frases)")
    trainer_autoreg.train()
    print(f"Ciclo {ciclo + 1}: época de SFT (50 preguntas)")
    trainer_sft.train()

In [ ]:
# Federico solo aparece en el texto plano: es otra de las 100 personas del examen
print(preguntar(model_C, "¿Cuál es el teléfono de Federico?"))

**Resultado obtenido** (en la ejecución original el entrenamiento se detuvo durante el séptimo ciclo):

```
<s>[INST] ¿Cuál es el teléfono de Federico? [/INST] El teléfono de Federico es el 928204525.
```

Vuelve a la tabla del apartado 1: **928204525 es, efectivamente, el teléfono de Federico**.

Recuerda lo que el modelo sabía de Federico: una única frase, `Federico tiene el número 928204525.`, perdida entre otras 149. Nunca vio una pregunta sobre él ni una respuesta con su nombre. Y sin embargo ha recuperado el dato y lo ha expresado con un formato («El teléfono de Federico es el...») que solo aparece en el otro fichero.

Y lo ha conseguido con **muchas menos épocas** que el experimento B: unas siete pasadas por el texto frente a treinta.

<div align="center">
    <img src="./imgs/agenda_estrategias.svg" width="850px">
</div>

## **6. ¿Por qué funciona mejor alternar?**

Los datos son los mismos en los tres experimentos. Lo único que cambia es **el orden** en el que el modelo los ve. Estas son las razones que mejor explican la diferencia.

### 6.1. El modelo aprende la *relación* entre los dos formatos

Para las 50 personas que están en ambos ficheros, el modelo ve, en ciclos consecutivos, el mismo hecho con dos caras:

```
Texto:     El número de Ariadna es 619609877.
Pregunta:  [INST] Dame el teléfono de Ariadna [/INST] El número de teléfono de Ariadna es el 619609877.
```

Al alternar, la forma más económica de reducir las dos pérdidas a la vez no es memorizar las frases por un lado y las respuestas por otro, sino aprender **una sola representación del hecho** que sirva para ambas, junto con la regla que las une: *lo que se dice de una persona en el texto es lo que hay que responder cuando preguntan por ella*. Esa regla no menciona a nadie en concreto, así que se aplica también a las 100 personas por las que nunca se preguntó. Las 50 personas compartidas actúan como **ejemplos resueltos** de cómo pasar del texto a la respuesta.

En el experimento B esto no puede ocurrir: cuando llegan las preguntas, el texto ya no está. El modelo solo puede memorizar las 50 respuestas.

### 6.2. Los hechos se repasan continuamente

Cada ciclo vuelve a pasar por las 150 frases. Lo que una época de SFT haya podido deteriorar, la siguiente época autorregresiva lo refuerza. Es el mismo principio que el **repaso espaciado** al estudiar: es más eficaz volver a la materia a intervalos que leerla treinta veces seguidas y no volver a verla. En aprendizaje automático esta técnica se conoce como *rehearsal* o *replay*, y es el remedio clásico contra el olvido catastrófico.

### 6.3. El modelo sabe desde el principio *para qué* va a usar lo que lee

En el experimento A el modelo almacena los hechos "sin saber" que después le preguntarán por ellos, y los guarda de la forma que mejor sirve para continuar frases. En el experimento C, desde el primer ciclo recibe la señal de que esos hechos hay que poder recuperarlos a partir de una pregunta, y los codifica de una manera que lo permite. Importa cómo se almacena un dato, no solo que se almacene.

### Una analogía

- **A** es leerse el temario sin hacer nunca un ejercicio: te suena todo, pero no sabes responder en el examen.
- **B** es leerse el temario entero y, semanas después, memorizar las soluciones de 50 ejercicios: respondes con mucha soltura, pero a las preguntas nuevas contestas con la solución de otro ejercicio.
- **C** es estudiar un tema y hacer ejercicios de ese tema, una y otra vez: aprendes a *usar* lo que has leído, también en preguntas que no habías visto.

### Lo que dice la investigación

Este fenómeno se ha estudiado de forma controlada, con biografías sintéticas en lugar de teléfonos. Allen-Zhu y Li (2023) observan justo lo que hemos visto aquí: que un modelo almacene un dato no garantiza que sepa *extraerlo* cuando se le pregunta; que preentrenar con el texto y afinar después con preguntas generaliza mal a las personas no vistas en las preguntas; y que entrenar mezclando ambos formatos, o presentar cada hecho redactado de varias maneras, mejora mucho esa extracción.

## **7. Midámoslo bien**

Hasta aquí hemos probado **una pregunta por experimento**, y además con personas distintas. Sirve para ilustrar, pero no para concluir: con un solo ejemplo no sabemos si el modelo C acierta siempre o si hemos tenido suerte.

La forma correcta de evaluar es preguntar por **todas** las personas y medir la tasa de acierto por separado en los dos grupos:

- las **50 vistas en el SFT**: mide cuánto ha memorizado;
- las **100 que solo están en el texto**: mide si ha aprendido a recuperar hechos, que es lo que nos interesa.

Fíjate en que evaluamos con una formulación de la pregunta, pero podrían usarse varias, incluidas algunas que no aparezcan en el entrenamiento.

In [ ]:
def telefono_en_respuesta(salida):
    # Extrae el primer número de 9 cifras que aparece después de [/INST]
    respuesta = salida.split("[/INST]")[-1]
    m = re.search(r"\d{9}", respuesta)
    return m.group() if m else None


def tasa_de_acierto(model, nombres):
    aciertos = 0
    for nombre in nombres:
        salida = preguntar(model, f"¿Cuál es el teléfono de {nombre}?")
        aciertos += telefono_en_respuesta(salida) == agenda[nombre]
    return aciertos / len(nombres)


print(f"Personas vistas en el SFT:      {tasa_de_acierto(model_C, vistas_en_sft):.0%}")
print(f"Personas solo en el texto:      {tasa_de_acierto(model_C, solo_en_texto):.0%}")

## **8. Ejercicios propuestos**

1. **Evalúa los tres modelos** con la función anterior y completa esta tabla. ¿Se confirma lo que sugerían los ejemplos sueltos?

| | Vistas en el SFT (50) | Solo en el texto (100) |
|---|---|---|
| A. Solo autorregresivo | | |
| B. Secuencial | | |
| C. Alternado | | |

2. **¿Cuántos ciclos hacen falta?** Evalúa el modelo C al final de cada ciclo y representa la tasa de acierto de cada grupo. ¿Cuál de los dos grupos se aprende antes?
3. **Menos ejemplos resueltos.** Repite el experimento C usando solo 25, 10 o 5 preguntas en el SFT. ¿Cuántas hacen falta para que el modelo aprenda la regla?
4. **Redacta cada hecho de varias maneras.** Genera un fichero de texto en el que cada persona aparezca en tres frases distintas. ¿Mejora el experimento B?
5. **Mezcla en lugar de alternar.** Junta los dos ficheros en un único conjunto de datos barajado y entrena con él. ¿Se comporta como el experimento C?
6. **La pregunta inversa.** Pregunta al modelo C «¿De quién es el teléfono 928204525?». ¿Lo sabe? Busca información sobre la *reversal curse*.
7. **Preguntas fuera de la agenda.** Pregunta por una persona que no existe. ¿Qué responde? ¿Cómo le enseñarías a decir «no lo sé»?

## **9. Conclusiones**

- **El entrenamiento autorregresivo aporta los hechos; el SFT aporta la forma de responder.** Ninguno de los dos basta por separado.
- **Encadenarlos no garantiza que se combinen.** Un modelo puede responder con un formato impecable y un contenido falso. Evalúa siempre con ejemplos que no estén en el conjunto de SFT.
- **Alternar (o mezclar) los dos tipos de datos** enseña al modelo la relación entre lo que lee y lo que debe responder, y esa relación se generaliza a hechos por los que nunca se le preguntó.
- **Basta con unos pocos ejemplos resueltos.** No hace falta escribir preguntas sobre todo el dominio: aquí, 50 pares han servido para que el modelo aprenda a usar 150 hechos.
- **El diseño de la evaluación es parte del entrenamiento.** Sin separar las personas "vistas" de las "no vistas", los experimentos B y C habrían parecido igual de buenos.

Una última reflexión práctica: para una agenda de teléfonos real nadie entrenaría un modelo. Sería mucho más fiable buscar el dato y dárselo en el contexto (RAG) o permitirle consultar una base de datos (*function calling*, tema 8). El ajuste fino es la herramienta adecuada cuando lo que se quiere cambiar es el **comportamiento** del modelo o darle un conocimiento amplio y estable del dominio. La agenda es solo un laboratorio donde todo se puede comprobar.

### **Referencias**

- Allen-Zhu y Li (2023). [Physics of Language Models: Part 3.1, Knowledge Storage and Extraction](https://arxiv.org/abs/2309.14316)
- Berglund et al. (2023). [The Reversal Curse: LLMs trained on "A is B" fail to learn "B is A"](https://arxiv.org/abs/2309.12288)
- Dettmers et al. (2023). [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)